# Export compact SLAP2–HARP synchronization evidence

**Read-only:** resolves session assets via `VIPSessionRegistry` and creates a portable ZIP containing source-extraction line-index metadata, complete chunked continuity diagnostics of the *derived* voltage timestamps, DMD1/DMD2 comparison, HARP DI3 rising edges, behavior QC, and small raw fluorescence/dFF windows.

Unlike a full recording export, no `.dat`, full `.h5`, or `.mat` files are copied. No voltage traces are filtered or smoothed. Source line indices are preserved separately from downstream timestamps; no absolute source→HARP mapping is assumed.

**Prerequisite:** put `export_sync_audit_bundle.py` in `src/vip_slap2_analysis/metadata/` in your local clone. Run from your normal Python environment, with repository installed editable (`pip install -e .`) or let the notebook import the local `src/` directory.

Potentially long-running operation: a full audit scans all derived timestamp samples (not full fluorescence recordings).

In [ ]:
from pathlib import Path
import sys
import json
import zipfile
import pandas as pd

# Modify these two settings only if repo or shared data are in a different location.
REPO_ROOT = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis")
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")

# The original T0.3 case, followed by any additional sessions you want to audit.
SESSION_IDS = ["863774_2026-08-04_08-12-47"]

# A local folder is preferable: write the small completed ZIP here instead of on VAST.
OUTPUT_DIR = Path.home() / "Desktop" / "slap2_sync_audit_exports"

INCLUDE_GLUTAMATE_CALCIUM_SOURCE_METADATA = True
EXPORT_HARP_DI3_RISING_EDGES = True
FULL_DERIVED_TIMEBASE_SCAN = True
OVERWRITE_EXPORT = False

SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.is_dir() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.metadata.export_sync_audit_bundle import export_session

assert SUMMARY_XLSX.is_file(), f"Missing workbook: {SUMMARY_XLSX}"
registry = VIPSessionRegistry.from_excel(SUMMARY_XLSX)
print(f"Registry: {len(registry.sessions_df)} sessions")
for session_id in SESSION_IDS:
    asset = registry.resolve_assets(session_id)
    print(session_id, '->', asset.session_dir, 'exists:', asset.session_dir.is_dir())

In [ ]:
# Optional: inspect other sessions in the registry before adding them to SESSION_IDS.
fields = [c for c in ["session_id", "subject_id", "session_type", "paradigm", "indicator1", "session_dir"] if c in registry.sessions_df.columns]
display(registry.sessions_df.loc[registry.sessions_df["subject_id"].isin([863774]), fields].head(20))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exports = []
for session_id in SESSION_IDS:
    asset = registry.resolve_assets(session_id)
    if not asset.session_dir.is_dir():
        raise FileNotFoundError(f"Cannot access session directory: {asset.session_dir}")
    output = OUTPUT_DIR / f"sync_audit_{session_id}.zip"
    print(f"Exporting {session_id} -> {output}")
    result = export_session(
        asset, output, roi_limit=2, boundary_samples=32,
        trace_window_samples=2048, chunk=262144,
        skip_full_scan=not FULL_DERIVED_TIMEBASE_SCAN,
        include_harp_edges=EXPORT_HARP_DI3_RISING_EDGES,
        include_glu_ca=INCLUDE_GLUTAMATE_CALCIUM_SOURCE_METADATA,
        overwrite=OVERWRITE_EXPORT,
    )
    exports.append(result)
    print('ZIP bytes:', result['zip_bytes'])
    if result['status']:
        print('Missing/failed optional assets:', json.dumps(result['status'], indent=2))

In [ ]:
# Verify ZIP integrity and display archive contents before sharing.
for r in exports:
    p = Path(r['output'])
    with zipfile.ZipFile(p, 'r') as z:
        bad = z.testzip()
        manifest = json.loads(z.read('manifest.json'))
        print(p.name, 'integrity:', 'PASS' if bad is None else bad)
        print('Entries:', len(z.namelist()), 'and processing notes:', manifest['notes'])
        if 'derived/voltage_audit.json' in z.namelist():
            audit = json.loads(z.read('derived/voltage_audit.json'))
            print('DMD comparison:', audit['dmd_comparison'])
            for dmd, rec in audit['dmds'].items():
                print(dmd, 'continuity:', {k: rec.get('timebase_scan', {}).get(k) for k in
                    ('n_samples_scanned', 'n_nonpositive_steps_within_epoch', 'n_uneven_steps_within_epoch', 'n_epoch_transitions')})